# M10 lab solution: Why RAG & the RAG Lifecycle

**Brief.** Baseline pipeline: build the canonical lifecycle notebook end to end on the course corpus and save its retrieval and answer metrics. Every later module reports its improvement against this baseline.

Run with the default environment: `uv run jupyter lab`. It writes `baseline.json` beside this notebook: the course's reference baseline, which every later lab reports against.

Open this notebook from `lab/solution/`, so `../data/` resolves to the lab's own data folder.

## Setup

In [ ]:
import json
import sys
import time
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()  # API keys from the .env file at the repo root
sys.path.insert(0, str(Path("../../../../src").resolve()))

from lab_kit import baseline, hit_rate, is_hit, load_handbook, load_jsonl, mrr, recall_at_k  # noqa: E402

DATA = Path("../data/larkspur")
handbook = load_handbook(DATA / "handbook")
golden = load_jsonl(DATA / "golden.jsonl")
by_id = {g["id"]: g for g in golden}
scored = [g for g in golden if g["evidence_spans"]]
print(len(handbook), "handbook files,", len(golden), "golden questions,", len(scored), "with evidence")

In [ ]:
from helpers import get_embeddings, get_experientiallabs_llm

llm = get_experientiallabs_llm()
embeddings = get_embeddings()
LLM_NAME = llm.model_name
EMBEDDING_NAME = getattr(embeddings, "model", None) or getattr(embeddings, "endpoint", None)
print("LLM:", LLM_NAME, "| embeddings:", EMBEDDING_NAME)

## 1. Index: load, split, embed, store

The same offline phase as `05a_rag_lifecycle_and_baseline.ipynb`, on the Larkspur handbook instead of the Bella Vista text. Loading is `load_handbook()` (one record per Markdown file); splitting matches `lab_kit.baseline` so the TF-IDF reference row and this pipeline see the same 30 chunks.

In [ ]:
# The inline "Part V system", reused by the later labs: the handbook split as lab_kit.baseline splits it
# (recursive, 500 characters, 50 overlap), a FAISS vector retriever over those chunks, and an answer
# prompt that cites numbered passages and is allowed to say "I don't know".
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = [
    Document(page_content=c, metadata={"source": p["source"], "section": p["title"]})
    for p in handbook
    for c in splitter.split_text(p["text"])
]
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "You answer questions from Larkspur Cloud customers using only the numbered handbook passages. "
               "Cite the passages you use as [1], [2]. If the passages do not contain the answer, "
               "reply exactly: I don't know."),
    ("human", "Passages:\n{context}\n\nQuestion: {question}"),
])


def format_docs(docs):
    return "\n\n".join(f"[{i}] ({d.metadata.get('source', '?')}) {d.page_content}" for i, d in enumerate(docs, 1))


def answer(question, docs, config=None):
    return (ANSWER_PROMPT | llm).invoke({"context": format_docs(docs), "question": question}, config=config).content


print(len(chunks), "chunks indexed")

In [ ]:
vector = embeddings.embed_query("How long does a refund take?")
print("embedding dimensions:", len(vector))
print("first chunk:", chunks[0].metadata, chunks[0].page_content[:120])

## 2. Retrieve: top 4 for every golden question

Retrieval metrics come from `lab_kit` and count only the 25 questions that have evidence spans. The TF-IDF row is `lab_kit.baseline()` on the same corpus, for reference.

In [ ]:
def texts(docs):
    return [d.page_content for d in docs]


def score(results, questions=None):
    """results: question id -> retrieved chunk texts in rank order; scored with lab_kit on the questions with evidence."""
    qs = questions or golden
    return {"hit_rate": round(hit_rate(results, qs), 3), "recall_at_k": round(recall_at_k(results, qs), 3),
            "mrr": round(mrr(results, qs), 3)}

In [ ]:
K = 4
retrieved, latency = {}, {}
for g in golden:
    t0 = time.perf_counter()
    retrieved[g["id"]] = retriever.invoke(g["question"])
    latency[g["id"]] = time.perf_counter() - t0

retrieval = score({qid: texts(docs) for qid, docs in retrieved.items()})
tfidf = baseline(DATA, k=K)
print("vector retriever:", retrieval)
print("TF-IDF reference:", {k: round(v, 3) for k, v in tfidf.items() if k in ("hit_rate", "recall_at_k", "mrr")})

## 3. Generate and judge

Every question is answered from its top 4 chunks. The 25 questions with evidence are graded by an LLM judge for correctness (against `reference_answer`) and faithfulness (against the passages). The 5 `out_of_corpus` questions are not judged: a reply that starts with "I don't know" counts as a refusal, which is the right answer for them. The judge prompt is printed first.

In [ ]:
from pydantic import BaseModel, Field


class Grade(BaseModel):
    correct: bool = Field(description="True when the answer states every fact the reference answer gives for this question.")
    faithful: bool = Field(description="True when every claim in the answer is supported by the passages.")


JUDGE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "You grade one answer from a customer-support RAG system. Judge it against the reference answer "
               "and the passages the system was given. Ignore citation markers such as [1]."),
    ("human", "Question: {question}\n\nReference answer: {reference}\n\nPassages:\n{context}\n\nAnswer to grade: {answer}"),
])
judge = JUDGE_PROMPT | llm.with_structured_output(Grade)


def grade(g, docs, reply):
    return judge.invoke({"question": g["question"], "reference": g["reference_answer"],
                         "context": format_docs(docs), "answer": reply})


def is_refusal(reply):
    return reply.strip().lower().startswith("i don't know")

In [ ]:
JUDGE_PROMPT.pretty_print()

In [ ]:
rows = []
for g in golden:
    docs = retrieved[g["id"]]
    t0 = time.perf_counter()
    reply = answer(g["question"], docs)
    gen_s = time.perf_counter() - t0
    row = {"id": g["id"], "type": g["type"], "refused": is_refusal(reply), "retrieve_s": round(latency[g["id"]], 3),
           "generate_s": round(gen_s, 3), "answer": reply}
    if g["evidence_spans"]:
        verdict = grade(g, docs, reply)
        row.update(correct=verdict.correct, faithful=verdict.faithful)
    rows.append(row)
    print(g["id"], g["type"], "| refused" if row["refused"] else "", "|", reply[:100].replace("\n", " "))

## 4. Results and the saved baseline

In [ ]:
import datetime

import pandas as pd

df = pd.DataFrame(rows)
in_corpus = df[df["type"] != "out_of_corpus"]
ooc = df[df["type"] == "out_of_corpus"]
answers = {
    "correctness": round(in_corpus["correct"].mean(), 3),
    "faithfulness": round(in_corpus["faithful"].mean(), 3),
    "refusals_out_of_corpus": f"{int(ooc['refused'].sum())}/{len(ooc)}",
    "refusals_in_corpus": f"{int(in_corpus['refused'].sum())}/{len(in_corpus)}",
    "mean_retrieve_s": round(df["retrieve_s"].mean(), 3),
    "mean_generate_s": round(df["generate_s"].mean(), 3),
}
result = {
    "date": datetime.date.today().isoformat(),
    "llm": LLM_NAME,
    "embeddings": EMBEDDING_NAME,
    "pipeline": {"chunking": "recursive, 500 characters, 50 overlap", "chunks": len(chunks), "store": "FAISS", "k": K},
    "retrieval": retrieval,
    "tfidf_reference": {k: round(v, 3) for k, v in tfidf.items()},
    "answers": answers,
    "by_type": df.groupby("type")[["correct", "faithful", "refused"]].mean().round(3).to_dict(orient="index"),
}
Path("baseline.json").write_text(json.dumps(result, indent=2) + "\n")
print(pd.DataFrame([{**retrieval, **answers}]).T.rename(columns={0: "value"}))
print(df.groupby("type")[["correct", "faithful", "refused"]].mean().round(3))

## What this shows

- The table is the whole lifecycle in numbers: retrieval quality first (hit rate, recall and MRR at k = 4), then what the model made of the passages (correctness, faithfulness), then how it behaves when the handbook has no answer (refusals on the 5 out-of-corpus questions, and false refusals on the 25 others).
- Compare the vector row with the TF-IDF reference: where they differ, the cause is the embedding model, because both read the same 30 chunks.
- A wrong answer with a retrieval hit points at generation; a wrong answer with no hit points at indexing or retrieval. The per-question rows above say which.
- `baseline.json` freezes these numbers with the date and models. Later labs change one stage and compare against it.